# Normalización y EDA de impagos

Este notebook:

- Carga `Variable Y.csv`
- Convierte `VERDADERO` a `1` y `FALSO` a `0`
- Trata valores vacíos y `#¡NULO!` como ausentes
- Calcula el porcentaje de impagos total, por mes/año y por año

In [ ]:
# Ejecuta esta celda si tu entorno no tiene las librerías instaladas.
%pip install pandas numpy matplotlib seaborn openpyxl xlrd

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

ruta_datos = Path('Variable Y.csv')

df = pd.read_csv(ruta_datos, sep=';', encoding='utf-8-sig')

# El archivo trae columnas sin nombre o vacías extra; se eliminan.
df = df.loc[:, ~df.columns.astype(str).str.match(r'^Unnamed')]
df = df.loc[:, df.columns.astype(str).str.strip() != '']

# Además, se eliminan columnas completamente vacías por seguridad.
df = df.dropna(axis=1, how='all')

# Si hubiera filas completamente vacías, también se eliminan.
df = df.dropna(axis=0, how='all').copy()

print(f'Filas: {df.shape[0]}, columnas: {df.shape[1]}')
df.head()

In [ ]:
df_normalizado = df.copy()

columnas_periodo = [col for col in df_normalizado.columns if col != 'ID']

mapeo_binario = {
    'VERDADERO': 1,
    'FALSO': 0,
    '#¡NULO!': np.nan,
    '': np.nan
}

df_normalizado[columnas_periodo] = df_normalizado[columnas_periodo].replace(mapeo_binario)
df_normalizado['ID'] = pd.to_numeric(df_normalizado['ID'], errors='coerce')

# Convierte las columnas mensuales a tipo numérico para asegurar 0/1.
df_normalizado[columnas_periodo] = df_normalizado[columnas_periodo].apply(pd.to_numeric, errors='coerce')

df_normalizado.head()

In [ ]:
# Exportación opcional del resultado normalizado.
ruta_salida = Path('Variable_Y_normalizada.csv')
df_normalizado.to_csv(ruta_salida, index=False)
print(f'Archivo guardado en: {ruta_salida.resolve()}')

In [ ]:
meses_es = {
    'Enero': '01',
    'Febrero': '02',
    'Marzo': '03',
    'Abril': '04',
    'Mayo': '05',
    'Junio': '06',
    'Julio': '07',
    'Agosto': '08',
    'Septiembre': '09',
    'Octubre': '10',
    'Noviembre': '11',
    'Diciembre': '12'
}

df_largo = df_normalizado.melt(
    id_vars='ID',
    value_vars=columnas_periodo,
    var_name='periodo',
    value_name='impago'
)

periodo_split = df_largo['periodo'].str.extract(r'(?P<mes>\w+)\s+(?P<anio>\d{4})')
df_largo['mes'] = periodo_split['mes']
df_largo['anio'] = pd.to_numeric(periodo_split['anio'], errors='coerce')
df_largo['fecha'] = pd.to_datetime(
    periodo_split['anio'].astype(str) + '-' + periodo_split['mes'].map(meses_es) + '-01',
    errors='coerce'
)

porcentaje_total = df_largo['impago'].mean() * 100

porcentaje_por_mes_anio = (
    df_largo.dropna(subset=['fecha'])
    .groupby('fecha', as_index=False)['impago']
    .mean()
    .assign(porcentaje_impago=lambda x: x['impago'] * 100)
    .sort_values('fecha')
)

porcentaje_por_anio = (
    df_largo.dropna(subset=['anio'])
    .groupby('anio', as_index=False)['impago']
    .mean()
    .assign(porcentaje_impago=lambda x: x['impago'] * 100)
    .sort_values('anio')
)

print(f'Porcentaje total de impagos: {porcentaje_total:.2f}%')

display(
    porcentaje_por_mes_anio[['fecha', 'porcentaje_impago']]
    .rename(columns={'fecha': 'mes_anio', 'porcentaje_impago': 'pct_impago'})
)

display(
    porcentaje_por_anio[['anio', 'porcentaje_impago']]
    .rename(columns={'porcentaje_impago': 'pct_impago'})
)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.barplot(
    data=porcentaje_por_mes_anio,
    x='fecha',
    y='porcentaje_impago',
    color='#D55E00',
    ax=axes[0]
)
axes[0].set_title('Porcentaje de impagos por mes/año')
axes[0].set_xlabel('Mes/Año')
axes[0].set_ylabel('% de impago')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(
    data=porcentaje_por_anio,
    x='anio',
    y='porcentaje_impago',
    color='#0072B2',
    ax=axes[1]
)
axes[1].set_title('Porcentaje de impagos por año')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('% de impago')

plt.tight_layout()
plt.show()